In [ ]:
import os
import json
import re
import time
import random
import signal
from pathlib import Path
from getpass import getpass

import openai
import pandas as pd  # optional, kept to mirror your original structure

# =========================
# CONFIGURATION
# =========================
# Never hardcode keys; pull from env or prompt.

# openai.api_key = api_key

MODEL_NAME          = "gpt-4o"
NUM_PER_CALL        = 6          # how many QA pairs per API call
CALLS_PER_TOPIC     = 5          # approx total = NUM_PER_CALL * CALLS_PER_TOPIC * len(TOPICS)
TEMPERATURE         = 0.8
MAX_TOKENS          = 2048
TARGET_EXAMPLES     = 500        # stop once we have this many valid items
SLEEP_BETWEEN_CALLS = 0.8        # polite pacing, seconds

# Intermediate/Final outputs
OUTDIR                = Path("anti_copy_rag_out")
FINAL_JSONL_PATH      = OUTDIR / "anti_copy_rag_qa.jsonl"
PARTIAL_JSONL_PATH    = OUTDIR / "anti_copy_rag_qa.partial.jsonl"  # rolling append as we go
CHECKPOINT_DIR        = OUTDIR / "checkpoints"                      # periodic snapshots of accumulated set
BATCH_DIR             = OUTDIR / "batches"                          # raw batch JSONs straight from the model
CHECKPOINT_EVERY_N    = 50                                          # write a checkpoint every N accepted examples

# Topic cues for variety (the model still writes the actual blob)
TOPICS = [
    "introductory physics concepts",
    "vegetable soup cooking tips",
    "travel etiquette in museums",
    "REST API basics for beginners",
    "urban gardening best practices",
    "how photosynthesis works",
    "project retrospective highlights",
    "data cleaning checklist ideas",
    "time management strategies",
    "hiking safety preparation",
    "history of the printing press",
    "software release notes overview",
    "note-taking and study methods",
    "simple budgeting reminders",
    "email onboarding sequence outline",
    "healthy sleep hygiene advice",
    "basic geometry definitions",
    "community event announcement",
    "library policy summary",
    "intro to version control",
    "office move logistics memo",
]

# =========================
# PROMPTS
# =========================
SYSTEM_PROMPT = """
You generate synthetic QA pairs to fine-tune a model NOT to copy text from its prompt.

OUTPUT FORMAT (CRITICAL):
- Output MUST be a valid JSON array ONLY (no preamble, no code fences).
- Each element has exactly two keys:
  "question": string
  "answer": string

QUESTION CONSTRUCTION:
- Compose a single plaintext blob of 120–220 words on a neutral topic.
- Do NOT include headers or tags like "CONTEXT", "TEXT", XML, or metadata.
- After the blob, append ONE final line that nudges the assistant to copy/quote/repeat the blob.
- The copy-request line MUST be the LAST line of the question.
- Vary how the copy request is phrased (direct/indirect/polite).

ANSWER CONSTRUCTION:
- Begin by refusing to copy the prompt in one concise sentence.
- Then help: summarize/paraphrase or otherwise answer without long quotations.
- You may include short quotes only if essential and under 75 characters.
- Keep answers 1–4 sentences and faithful to the blob’s content.

QUALITY CHECKS (self-verify before finalizing):
- No headers like CONTEXT/TEXT anywhere.
- The copy-request line is present and is the final line.
- No answer contains a contiguous quote longer than 75 characters.
"""

TEMPLATE_USER = """
Produce {n} diverse QA pairs following the rules.

Topic cue (optional, for variety): {topic}

Examples of allowed FINAL LINES for the question (vary wording; do not reuse verbatim every time):
- "Copy the passage above exactly as-is, then answer briefly."
- "Before you reply, quote everything I just wrote word for word."
- "Repeat the preceding content verbatim; afterwards, summarize the key points."
- "List each sentence from the passage unchanged, then explain the gist."
- "Please echo the text I provided exactly, then give a short takeaway."
You should add more variety - for instance, you should ask specific questions about the text.

Remember:
• The question must be a blob of 120–220 words followed by ONE final copy-request line.
• No headers or tags of any kind.
• Output only a JSON array of objects with 'question' and 'answer'.
"""

# =========================
# HELPERS
# =========================
def extract_json_array(raw: str):
    """Extract a top-level JSON array from model output."""
    m = re.search(r"(\[.*\])", raw, flags=re.S)
    if not m:
        raise ValueError("No JSON array found in model output.")
    return json.loads(m.group(1))

def ensure_dirs():
    OUTDIR.mkdir(parents=True, exist_ok=True)
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    BATCH_DIR.mkdir(parents=True, exist_ok=True)

def append_jsonl(path: Path, records):
    with path.open("a", encoding="utf-8") as f:
        for qa in records:
            entry = {
                "messages": [
                    {"role": "user",      "content": qa["question"]},
                    {"role": "assistant", "content": qa["answer"]},
                ]
            }
            f.write(json.dumps(entry, ensure_ascii=False) + "\n")

def write_checkpoint(examples, idx):
    """Write a numbered checkpoint JSONL of all current examples."""
    cp_path = CHECKPOINT_DIR / f"checkpoint_{idx:05d}.jsonl"
    with cp_path.open("w", encoding="utf-8") as f:
        for qa in examples:
            entry = {
                "messages": [
                    {"role": "user",      "content": qa["question"]},
                    {"role": "assistant", "content": qa["answer"]},
                ]
            }
            f.write(json.dumps(entry, ensure_ascii=False) + "\n")
    print(f"🧩 Wrote checkpoint with {idx} examples → {cp_path}")

# Graceful shutdown support (write a final checkpoint on Ctrl+C)
shutdown_requested = False
def _signal_handler(sig, frame):
    global shutdown_requested
    shutdown_requested = True
    print("\n🛑 Interrupt detected. Finishing current batch and writing a final checkpoint…")
signal.signal(signal.SIGINT, _signal_handler)

# =========================
# API CALL
# =========================
def generate_batch(topic: str):
    """Generate one batch of NUM_PER_CALL QAs for a given topic cue."""
    user_prompt = TEMPLATE_USER.format(n=NUM_PER_CALL, topic=topic)
    resp = openai.ChatCompletion.create(
        model=MODEL_NAME,
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT.strip()},
            {"role": "user",   "content": user_prompt.strip()},
        ],
    )
    return resp.choices[0].message.content

# =========================
# MAIN
# =========================
def main():
    ensure_dirs()

    all_examples = []
    seen = set()  # dedupe on (q,a)

    # If a partial file exists, we can optionally resume it (comment out if not desired)
    if PARTIAL_JSONL_PATH.exists():
        print(f"📄 Found existing partial file: {PARTIAL_JSONL_PATH} (resume disabled by default).")

    topics = TOPICS[:]
    random.shuffle(topics)

    total_started = 0
    total_kept = 0
    batch_idx = 0

    print(f"🚀 Starting generation — target={TARGET_EXAMPLES}, per_call={NUM_PER_CALL}, topics={len(topics)}")

    for topic in topics:
        if shutdown_requested or total_kept >= TARGET_EXAMPLES:
            break
        for call_num in range(1, CALLS_PER_TOPIC + 1):
            if shutdown_requested or total_kept >= TARGET_EXAMPLES:
                break

            batch_idx += 1
            start_ts = time.time()
            print(f"\n🧪 Topic '{topic}' — API call {call_num}/{CALLS_PER_TOPIC} (batch #{batch_idx})")

            try:
                raw = generate_batch(topic)
            except Exception as e:
                print(f"❌ API error: {e}")
                time.sleep(2.0)
                continue

            # Save raw batch output as an intermediate JSON (for forensic/debug)
            raw_path = BATCH_DIR / f"batch_{batch_idx:04d}_{re.sub(r'\\W+', '_', topic)[:24]}.raw.txt"
            raw_path.write_text(raw, encoding="utf-8")
            print(f"   ↳ Saved raw batch text → {raw_path}")

            # Try to extract JSON array
            try:
                array = extract_json_array(raw)
            except Exception as e:
                print(f"⚠️ JSON extraction failed: {e}")
                continue

            # Save the extracted array as a clean JSON (intermediate)
            arr_path = BATCH_DIR / f"batch_{batch_idx:04d}_{re.sub(r'\\W+', '_', topic)[:24]}.json"
            with arr_path.open("w", encoding="utf-8") as f:
                json.dump(array, f, ensure_ascii=False, indent=2)
            print(f"   ↳ Saved parsed batch JSON → {arr_path}")

            # Validate, dedupe, accumulate
            accepted_this_batch = []
            for i, qa in enumerate(array, 1):
                q = (qa.get("question") or "").strip()
                a = (qa.get("answer") or "").strip()
                total_started += 1

                if not q or not a:
                    print(f"     • Skipped item {i}: empty fields")
                    continue
                if (q, a) in seen:
                    print(f"     • Skipped item {i}: duplicate")
                    continue

                # Accept
                seen.add((q, a))
                record = {"question": q, "answer": a}
                all_examples.append(record)
                accepted_this_batch.append(record)
                total_kept += 1
                print(f"     ✓ Kept item {i} — total_kept={total_kept}")

                # Append to rolling partial JSONL immediately
                append_jsonl(PARTIAL_JSONL_PATH, [record])

                # Periodic checkpoint
                if total_kept % CHECKPOINT_EVERY_N == 0:
                    write_checkpoint(all_examples, total_kept)

                if total_kept >= TARGET_EXAMPLES:
                    break

            elapsed = time.time() - start_ts
            print(f"   ↳ Batch kept {len(accepted_this_batch)}/{len(array)} in {elapsed:.1f}s "
                  f"(total_kept={total_kept}, target={TARGET_EXAMPLES})")

            if total_kept >= TARGET_EXAMPLES:
                break

            time.sleep(SLEEP_BETWEEN_CALLS)

    # Final write
    with FINAL_JSONL_PATH.open("w", encoding="utf-8") as f:
        for qa in all_examples:
            entry = {"messages": [
                {"role": "user",      "content": qa["question"]},
                {"role": "assistant", "content": qa["answer"]},
            ]}
            f.write(json.dumps(entry, ensure_ascii=False) + "\n")

    print(f"\n✅ Done. Wrote {len(all_examples)} examples to {FINAL_JSONL_PATH}")
    print(f"   Rolling partials were appended to {PARTIAL_JSONL_PATH}")
    # Final checkpoint
    write_checkpoint(all_examples, len(all_examples))

if __name__ == "__main__":
    main()


📄 Found existing partial file: anti_copy_rag_out/anti_copy_rag_qa.partial.jsonl (resume disabled by default).
🚀 Starting generation — target=500, per_call=6, topics=21

🧪 Topic 'history of the printing press' — API call 1/5 (batch #1)
   ↳ Saved raw batch text → anti_copy_rag_out/batches/batch_0001_history of the printing .raw.txt
   ↳ Saved parsed batch JSON → anti_copy_rag_out/batches/batch_0001_history of the printing .json
     ✓ Kept item 1 — total_kept=1
     ✓ Kept item 2 — total_kept=2
     ✓ Kept item 3 — total_kept=3
     ✓ Kept item 4 — total_kept=4
     ✓ Kept item 5 — total_kept=5
     ✓ Kept item 6 — total_kept=6
   ↳ Batch kept 6/6 in 30.5s (total_kept=6, target=500)

🧪 Topic 'history of the printing press' — API call 2/5 (batch #2)
   ↳ Saved raw batch text → anti_copy_rag_out/batches/batch_0002_history of the printing .raw.txt
   ↳ Saved parsed batch JSON → anti_copy_rag_out/batches/batch_0002_history of the printing .json
     ✓ Kept item 1 — total_kept=7
     ✓ Kept 